# StockSense - Round 3: Intelligence Layer, Recommendations & What-If

Pipeline for every store x product on the latest date:

`forecast -> stock-out probability -> risk band -> reasons -> reorder quantity -> manager action`

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from IPython.display import Image, Markdown, display
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)
FIG = ROOT / 'reports' / 'figures'

In [2]:
from src.recommend import score_latest, load_models
from src.config import SERVICE_Z
feat = pd.read_csv(ROOT / 'data' / 'processed' / 'features.csv', parse_dates=['date'])
reg, clf = load_models()
rec = score_latest(feat, reg, clf)
print('As of', rec.as_of_date[0], '| forecast window: next 7 days')
rec.risk_level.value_counts()

As of 2026-08-31 | forecast window: next 7 days


risk_level
LOW       158
HIGH       97
MEDIUM     27
Name: count, dtype: int64

## 1. Business logic

| Item | Rule |
|---|---|
| Risk | HIGH if p >= 0.70, MEDIUM if 0.40 <= p < 0.70, else LOW |
| Safety stock | z x daily demand std x sqrt(lead time), z = 1.65 / 1.28 / 1.00 for High / Medium / Low |
| Recommended stock | 7-day forecast + safety stock (capped at shelf life for perishables) |
| Reorder quantity | max(0, recommended - current - incoming), rounded up to the case pack |
| Guard-rails | Empty shelf -> p >= 0.95; stock runs out before a new order can land -> p >= 0.70 |

The guard-rails exist because the model cannot see open purchase orders. It learned that empty
shelves are often refilled soon, which is statistically true but not actionable. Every escalated
item is labelled `Rule: ...` so managers can see exactly why.

## 2. Manager-ready output (format from the brief)

In [3]:
out = rec[['store_id','product_name','current_stock','forecast_7d','stockout_probability','risk_level','reorder_qty','manager_action']]
out.columns = ['Store','Product','Current Stock','7-Day Forecast','Stock-out Prob.','Risk','Recommended Order','Action']
out.head(15)

,Store,Product,Current Stock,7-Day Forecast,Stock-out Prob.,Risk,Recommended Order,Action
0,S02,Milk 1L,268,786.4,1.0000,HIGH,144,Raise replenishment order today; split into da...
1,S02,Detergent 1kg,0,202.4,0.9500,HIGH,288,Raise replenishment order today - SUP09 needs ...
2,S02,Basmati Rice 1kg,0,209.6,0.9500,HIGH,270,Raise replenishment order today - SUP05 needs ...
3,S02,Sunflower Oil 1L,0,180.4,0.9500,HIGH,252,Raise replenishment order today - SUP05 needs ...
4,S02,Toor Dal 1kg,0,119.8,0.9500,HIGH,174,Raise replenishment order today - SUP05 needs ...
5,S02,Paneer 200g,0,212.0,0.9500,HIGH,246,Raise replenishment order today
6,S02,Bath Soap 4pk,0,138.1,0.9500,HIGH,174,Raise replenishment order today - SUP08 needs ...
7,S02,Curd 500g,0,368.1,0.9500,HIGH,306,Raise replenishment order today; split into da...
8,S02,Mango Juice 1L,0,129.4,0.9500,HIGH,156,Raise replenishment order today - SUP04 needs ...
9,S02,Guava Juice 1L,0,126.6,0.9500,HIGH,168,Raise replenishment order today - SUP04 needs ...


## 3. Recommendation card

In [4]:
def card(r):
    reasons = ', '.join(n if v == 'rule' else f'{n} ({v})' for n, v in json.loads(r.reasons))
    print(f"STORE {r.store_id} - {r.product_name.upper()}")
    print(f"Predicted 7-day demand: {r.forecast_7d:,.0f} units")
    print(f"Current stock: {r.current_stock:,} units | Incoming stock: {r.incoming_stock} units")
    print(f"Stock-out probability: {r.stockout_probability:.0%} (model {r.model_probability:.0%}) | Risk: {r.risk_level}")
    print(f"Safety stock: {r.safety_stock:.0f} | Recommended stock: {r.recommended_stock:.0f}")
    print(f"Recommended additional replenishment: {r.reorder_qty} units (case pack {r.case_pack})")
    print(f"Why? {reasons}")
    print(f"MANAGER ACTION: {r.manager_action}\n")
for _, r in rec[rec.risk_level=='HIGH'].head(3).iterrows(): card(r)
card(rec[rec.risk_level=='MEDIUM'].iloc[0])

STORE S02 - MILK 1L
Predicted 7-day demand: 786 units
Current stock: 268 units | Incoming stock: 0 units
Stock-out probability: 100% (model 100%) | Risk: HIGH
Safety stock: 74 | Recommended stock: 412
Recommended additional replenishment: 144 units (case pack 6)
Why? No recent delivery (+18 pts risk), Low stock cover vs demand (+18 pts risk), Repeated recent stock-outs (+18 pts risk)
MANAGER ACTION: Raise replenishment order today; split into daily drops (short shelf life)

STORE S02 - DETERGENT 1KG
Predicted 7-day demand: 202 units
Current stock: 0 units | Incoming stock: 0 units
Stock-out probability: 95% (model 55%) | Risk: HIGH
Safety stock: 85 | Recommended stock: 288
Recommended additional replenishment: 288 units (case pack 6)
Why? Shelf already empty, No recent delivery (+4 pts risk), Low stock cover vs demand (+3 pts risk)
MANAGER ACTION: Raise replenishment order today - SUP09 needs 5 days, ask for express delivery

STORE S02 - BASMATI RICE 1KG
Predicted 7-day demand: 210 uni

## 4. Local explanations
For each item, each business driver is reset to a neutral value and both models are re-run. The
change in prediction is that driver's contribution. This is permutation logic applied to a single
row, and it reads naturally to a manager ("Promotion running: +10 pts risk").

In [5]:
r = rec[rec.risk_level=='HIGH'].iloc[0]
pd.DataFrame({'risk impact (prob. points)': json.loads(r.risk_drivers),
              'demand impact (%)': json.loads(r.demand_drivers)}).sort_values('risk impact (prob. points)', ascending=False)

,risk impact (prob. points),demand impact (%)
No recent delivery,18.5,0.2
Low stock cover vs demand,18.2,-2.0
Repeated recent stock-outs,18.2,0.4
Festival / holiday in next 7 days,5.3,0.5
Promotion running,0.0,-0.3
Recent sales growth,0.0,0.0
Long supplier lead time,0.0,34.9
Hot weather,0.0,-0.0
Unpredictable demand,0.0,0.5
Local event nearby,0.0,0.0


## 5. Overstock: the other half of the problem

In [6]:
over = rec[rec.excess_units > 0].sort_values('excess_value', ascending=False)
print(f"{len(over)} items hold more than 14 days of stock (or more than their shelf life): Rs {over.excess_value.sum():,.0f} at cost")
over[['store_id','product_name','current_stock','forecast_7d','days_of_cover','excess_value','manager_action']].head(10)

24 items hold more than 14 days of stock (or more than their shelf life): Rs 17,074 at cost


,store_id,product_name,current_stock,forecast_7d,days_of_cover,excess_value,manager_action
278,S06,Toor Dal 1kg,140,62.1,15.8,2384.0,Hold orders - overstocked; consider a promotio...
256,S03,Shampoo 180ml,70,23.1,21.2,1896.0,Hold orders - overstocked; consider a promotio...
279,S06,Ghee 500ml,50,22.5,15.6,1425.0,Hold orders - overstocked; consider a promotio...
38,S04,Milk 1L,142,267.6,3.7,1404.0,Order within 2 days and recheck the shelf tomo...
266,S05,Cola 750ml,274,117.0,16.4,1240.0,Hold orders - overstocked; consider a promotio...
237,S03,Cheese Slices 200g,28,9.2,21.3,1120.0,Hold orders - overstocked; consider a promotio...
267,S05,Protein Bar 60g,83,31.8,18.3,988.0,Hold orders - overstocked; consider a promotio...
275,S06,Cheese Slices 200g,67,29.6,15.8,896.0,Hold orders - overstocked; consider a promotio...
276,S05,Water 1L,309,124.1,17.4,671.0,Hold orders - overstocked; consider a promotio...
258,S02,Tissue Roll 4pk,111,53.0,14.7,630.0,Hold orders - overstocked; consider a promotio...


## 6. What-if analysis

In [7]:
base = rec
rows = []
for name, s in [('10% promotion', dict(discount_pct=10)), ('20% promotion', dict(discount_pct=20)),
                ('Supplier delay +2 days', dict(supplier_delay_days=2)), ('Local festival', dict(festival=1)),
                ('Demand shock +20%', dict(demand_shock_pct=20))]:
    x = score_latest(feat, reg, clf, scenario=s, explain_rows=False)
    rows.append(dict(scenario=name, demand_change=f"{x.forecast_7d.sum()/base.forecast_7d.sum()-1:+.1%}",
                     high_risk_items=int((x.risk_level=='HIGH').sum()), change=int((x.risk_level=='HIGH').sum()-(base.risk_level=='HIGH').sum()),
                     units_to_order=int(x.reorder_qty.sum())))
pd.DataFrame(rows)

,scenario,demand_change,high_risk_items,change,units_to_order
0,10% promotion,+4.9%,108,11,13980
1,20% promotion,+8.5%,113,16,14652
2,Supplier delay +2 days,+0.0%,156,59,14688
3,Local festival,+4.5%,103,6,13890
4,Demand shock +20%,+20.0%,111,14,16608


Supplier delay has by far the largest effect on risk: two extra days push roughly 60 more items
into High. Promotions raise demand about 5-9% and lift orders with it. The festival effect is
probably under-estimated, because there were only a handful of festival days to learn from. This
is a known limitation, listed in the README.

## 7. Save outputs

In [8]:
from src.recommend import OUTPUT_COLS
rec[OUTPUT_COLS].to_csv(ROOT / 'data' / 'processed' / 'recommendations.csv', index=False)
print('Saved data/processed/recommendations.csv  ->  open dashboard/stocksense_dashboard.html or run: streamlit run dashboard/app.py')

Saved data/processed/recommendations.csv  ->  open dashboard/stocksense_dashboard.html or run: streamlit run dashboard/app.py
